# Session 6: Data Restructuring

**Learning goals:**

- Merge datasets by matching one or more key columns.
- Concatenate compatible datasets by rows or columns.
- Group observations and calculate clear summary measures with `groupby()`.
- Select and reshape the `MultiIndex` produced by grouping on multiple columns.
- Reshape data between wide and long forms with `melt()`, `pivot()`, and `pivot_table()`.
- Move labels between columns and a hierarchical index with `stack()` and `unstack()`.

**Prerequisites:**

- Basic Python syntax.
- Introductory pandas skills, including loading and inspecting a `DataFrame`.


## Outline

0. **Loading data**: load the three case datasets
1. **Merging and concatenating**: combine related tables
2. **Grouping, aggregation, and MultiIndex**: summarise and organise observations
3. **Pivoting and melting**: move between wide and long forms
4. **Stacking and unstacking**: reshape data through the index
5. **Exercises**: practise the restructuring workflow
6. **Summary**: check results and choose an appropriate method


## 0. Loading Data

Load the three CSV case datasets directly from the project on GitHub.


In [1]:
# Import pandas and define the three source URLs.
import pandas as pd

STATE_TARGETS_URL = 'https://raw.githubusercontent.com/wlos/DA_with_Python/refs/heads/main/sample_data/state_targets.csv'
PRODUCT_LOOKUP_URL = 'https://raw.githubusercontent.com/wlos/DA_with_Python/refs/heads/main/sample_data/product_lookup.csv'
MARKETING_SPEND_URL = 'https://raw.githubusercontent.com/wlos/DA_with_Python/refs/heads/main/sample_data/marketing_spend_wide.csv'


Use `pd.read_csv()` to load the state targets, product lookup, and marketing spend datasets.


In [2]:
# Load the three CSV datasets.
targets = pd.read_csv(STATE_TARGETS_URL)  # Read the remote CSV into a DataFrame.
product_lookup = pd.read_csv(PRODUCT_LOOKUP_URL)  # Read product attributes by category.
marketing_wide = pd.read_csv(MARKETING_SPEND_URL)  # Read one marketing channel per column.

print('State targets:')
display(targets)
print('Product lookup:')
display(product_lookup)
print('Marketing spend in wide form:')
display(marketing_wide)


State targets:


,state,quarter,sales_target
0,WA,2025Q1,4800
1,WA,2025Q2,5300
2,NSW,2025Q1,5050
3,NSW,2025Q2,5550
4,VIC,2025Q1,4950
5,VIC,2025Q2,5450
6,QLD,2025Q1,4800
7,QLD,2025Q2,5300


Product lookup:


,category,manager,base_margin_pct,priority
0,Electronics,Nina Chen,0.32,High
1,Home,Luca Martin,0.28,Medium
2,Office,Samir Patel,0.25,High
3,Garden,Harper Stone,0.30,Low


Marketing spend in wide form:


,month,digital,catalogue,events
0,2025-01,1200,800,500
1,2025-02,1400,780,620
2,2025-03,1500,760,650
3,2025-04,1650,730,720
4,2025-05,1725,710,760
5,2025-06,1800,690,810


## 1. Merging and Concatenating Datasets

Both operations combine data, but they solve different problems.

| Method | Main purpose | How rows are aligned | Typical use |
| --- | --- | --- | --- |
| `merge()` | join related tables | matching values in one or more key columns | combine product details by category |
| `pd.concat(..., axis=0)` | stack objects vertically | shared column names | append quarterly extracts |
| `pd.concat(..., axis=1)` | place objects side by side | matching index labels | combine aligned measures |

### 1.1 Merge choices

| `how` value | Keys retained | Typical question |
| --- | --- | --- |
| `'inner'` | keys found in both tables | Which records have a match? |
| `'left'` | every left key | Can lookup details be added without losing left rows? |
| `'right'` | every right key | Which right-side records exist, whether matched or not? |
| `'outer'` | all keys from both tables | Which keys appear anywhere, including unmatched keys? |

The `validate` argument checks the expected key relationship before accepting a result.

| `validate` value | Expected relationship |
| --- | --- |
| `'one_to_one'` | each key occurs at most once in both tables |
| `'one_to_many'` | left keys are unique; right keys may repeat |
| `'many_to_one'` | left keys may repeat; right keys are unique |
| `'many_to_many'` | keys may repeat in both tables |

Unlike `merge()`, `concat()` does not match rows using a selected key column. With `axis=0`, it aligns columns by name; with `axis=1`, it aligns rows by index label.


### 1.2 Merging on a key

To demonstrate a keyed merge, treat the management and margin columns from the product lookup as two related extracts. Both contain one row per `category`, so `validate='one_to_one'` checks the expected relationship.


In [ ]:
# Create two related extracts from the product lookup.
product_management = product_lookup[['category', 'manager', 'priority']]  
product_margins = product_lookup[['category', 'base_margin_pct']] 

# Merge the extracts by their shared category key.
products_merged = product_management.merge(
    product_margins,        # Use the margin extract as the right-hand table.
    how='left',             # Keep every category from product_management.
    on='category',          # Match rows using the shared category key.
    validate='one_to_one',  # Confirm that each category occurs once in each table.
    indicator=True,         # Add a column showing whether each row matched.
)

products_merged


,category,manager,priority,base_margin_pct,_merge
0,Electronics,Nina Chen,High,0.32,both
1,Home,Luca Martin,Medium,0.28,both
2,Office,Samir Patel,High,0.25,both
3,Garden,Harper Stone,Low,0.30,both


The `_merge` column reports whether each row matched. Checking it is a quick way to detect missing lookup records before continuing.


In [4]:
# Count the merge outcomes to confirm that every category matched.
products_merged['_merge'].value_counts()


_merge
both          4
left_only     0
right_only    0
Name: count, dtype: int64

### 1.3 Concatenating rows

Suppose the targets arrive as separate quarterly extracts with the same columns. `pd.concat()` can stack them into one table. Dictionary keys provide an explicit source label.


In [ ]:
# Create two extracts with the same columns.
q1_targets = targets.loc[targets['quarter'] == '2025Q1'].copy() 
q2_targets = targets.loc[targets['quarter'] == '2025Q2'].copy() 

# Stack the extracts and retain their source.
targets_concatenated = (
    pd.concat(
        {'Q1 extract': q1_targets, 'Q2 extract': q2_targets},  # Tables and their source labels.
        names=['source'],  # Name the new outer index level.
    )
    .reset_index(level='source')  # Move the source label into a column.
    .reset_index(drop=True)  # Replace the old row indexes with a clean index.
)

targets_concatenated


,source,state,quarter,sales_target
0,Q1 extract,WA,2025Q1,4800
1,Q1 extract,NSW,2025Q1,5050
2,Q1 extract,VIC,2025Q1,4950
3,Q1 extract,QLD,2025Q1,4800
4,Q2 extract,WA,2025Q2,5300
5,Q2 extract,NSW,2025Q2,5550
6,Q2 extract,VIC,2025Q2,5450
7,Q2 extract,QLD,2025Q2,5300


The row count after concatenation should equal the sum of the input row counts. This simple check helps catch accidentally omitted extracts.


In [6]:
# Validate the expected number of concatenated rows.
expected_rows = len(q1_targets) + len(q2_targets)
assert len(targets_concatenated) == expected_rows
print(f'Validated {len(targets_concatenated)} concatenated rows.')


Validated 8 concatenated rows.


### 1.4 Concatenating columns

Use `axis=1` when tables should be placed side by side and their indexes already represent the same entities. Set `category` as the index in both product extracts before concatenating them.


In [ ]:
# Align both extracts by category before concatenating columns.
management_by_category = product_management.set_index('category')  # Use category labels for alignment.
margins_by_category = product_margins.set_index('category')  # Use the same index on the second table.

products_side_by_side = (
    pd.concat(
        [management_by_category, margins_by_category],
        axis=1,         # Place columns side by side.
        join='inner',   # Keep category labels found in both indexes.
    )
    .reset_index()      # Return category to an ordinary column.
)

products_side_by_side


,category,manager,priority,base_margin_pct
0,Electronics,Nina Chen,High,0.32
1,Home,Luca Martin,Medium,0.28
2,Office,Samir Patel,High,0.25
3,Garden,Harper Stone,Low,0.30


## 2. Grouping, Aggregation, and MultiIndex

A grouping workflow follows a **split–apply–combine** pattern:

1. split rows into groups using one or more columns
2. apply a calculation to each group
3. combine the results into a summary

| Aggregation | Meaning | Named-aggregation example |
| --- | --- | --- |
| `sum` | total within each group | `total_target=('sales_target', 'sum')` |
| `mean` | arithmetic average | `average_target=('sales_target', 'mean')` |
| `size` | number of rows, including missing values | `category_count=('category', 'size')` |
| `nunique` | number of distinct non-missing values | `manager_count=('manager', 'nunique')` |

Related index tools:

| Method | Purpose | Example |
| --- | --- | --- |
| `groupby(['a', 'b'])` | group by two columns and produce a `MultiIndex` result | group targets by state and quarter |
| `.xs(value, level=...)` | select one cross-section of a `MultiIndex` | select all quarters for WA |
| `.loc[(..., ...), :]` | select using several index levels | select Q2 for every state |
| `.unstack(level)` | move one index level into columns | create one quarter column per state |


Group state targets and apply several named aggregations. Setting `as_index=False` returns ordinary columns rather than moving the group key into the index.


In [ ]:
# Summarise the target values for each state.
state_target_summary = (
    targets.groupby('state', as_index=False)  # Form one group for each state.
    .agg(
        total_target=('sales_target', 'sum'),       # Add all quarterly targets.
        average_target=('sales_target', 'mean'),    # Calculate the mean quarterly target.
        number_of_quarters=('quarter', 'nunique'),  # Count distinct quarters.
    )
    .sort_values('total_target', ascending=False)
)

state_target_summary


,state,total_target,average_target,number_of_quarters
0,NSW,10600,5300.0,2
2,VIC,10400,5200.0,2
1,QLD,10100,5050.0,2
3,WA,10100,5050.0,2


The product lookup can also be grouped by priority to count categories and compare average base margins.


In [ ]:
# Summarise product categories within each priority group.
priority_summary = (
    product_lookup.groupby('priority', as_index=False, dropna=False)  # Keep priority as a column and retain missing groups.
    .agg(
        category_count=('category', 'size'),              # Count rows in each priority group.
        manager_count=('manager', 'nunique'),             # Count distinct managers.
        average_base_margin=('base_margin_pct', 'mean'),  # Calculate the mean margin.
    )
    .sort_values('average_base_margin', ascending=False)
)

priority_summary


,priority,category_count,manager_count,average_base_margin
1,Low,1,1,0.300
0,High,2,2,0.285
2,Medium,1,1,0.280


### 2.1 Grouping, Selecting a `MultiIndex`, and Unstacking

Grouping by more than one column produces a result with a `MultiIndex`. Here, `state` is the first index level and `quarter` is the second. After grouping, use `xs()` or `.loc` to select index values, then use `unstack('quarter')` to move the quarter index level into columns.


In [ ]:
# Group by two columns to produce a MultiIndex Series.
state_quarter_targets = (
    targets.groupby(['state', 'quarter'])['sales_target']  # Group by two keys and select the value column.
    .sum()          # Aggregate each state-quarter group to one value.
    .sort_index()   # Sort both index levels before selecting from them.
)

# xs() selects one value from a named index level.
wa_targets = state_quarter_targets.xs('WA', level='state')

# slice(None) means every state, while quarter is fixed to 2025Q2.
q2_targets_by_state = state_quarter_targets.loc[(slice(None), '2025Q2')]

# Move quarter labels from the row index into columns.
targets_by_state_quarter = state_quarter_targets.unstack('quarter', fill_value=0)

print('Grouped MultiIndex Series:')
display(state_quarter_targets.to_frame('total_target'))
print("WA only with xs('WA', level='state'):")
display(wa_targets.to_frame('total_target'))
print("Q2 only with loc[(slice(None), '2025Q2')]:")
display(q2_targets_by_state.to_frame('total_target'))
print('Unstacked state-by-quarter table:')
display(targets_by_state_quarter)


Grouped MultiIndex Series:


total_target
state quarter              
NSW   2025Q1           5050
      2025Q2           5550
QLD   2025Q1           4800
      2025Q2           5300
VIC   2025Q1           4950
      2025Q2           5450
WA    2025Q1           4800
      2025Q2           5300

WA only with xs('WA', level='state'):


,total_target
quarter,
2025Q1,4800
2025Q2,5300


Q2 only with loc[(slice(None), '2025Q2')]:


,total_target
state,
NSW,5550
QLD,5300
VIC,5450
WA,5300


Unstacked state-by-quarter table:


quarter,2025Q1,2025Q2
state,,
NSW,5050,5550
QLD,4800,5300
VIC,4950,5450
WA,4800,5300


## 3. Reshaping with `pivot()`, `pivot_table()`, and `melt()`

**Wide data** stores repeated measures in separate columns. **Long data** stores the measure name and value in separate variables. Long form is often easier to group, filter, and visualise.

| Method | Direction | Use it when | Duplicate combinations |
| --- | --- | --- | --- |
| `melt()` | wide to long | column names represent values of one variable | allowed |
| `pivot()` | long to wide | each index–column pair is unique | raises an error |
| `pivot_table()` | long to a summarised wide table | duplicate pairs need aggregation | handled by `aggfunc` |

Quick comparisons:

| Methods | Main difference |
| --- | --- |
| `melt()` vs `stack()` | `melt()` creates ordinary variable and value columns; `stack()` moves column labels into a `MultiIndex`. |
| `groupby().agg().unstack()` vs `pivot_table()` | the first exposes aggregation and reshaping as separate steps; the second performs both in one concise operation. |


### 3.1 Wide to long with `melt()`

The marketing table has one column per channel. Keep `month` as the identifier and melt the three channel columns into `channel` and `spend`.


In [ ]:
# Convert channel columns into tidy channel and spend variables.
marketing_long = marketing_wide.melt(
    id_vars='month',                                # Keep month as the identifier column.
    value_vars=['digital', 'catalogue', 'events'],  # Move these channel columns into rows.
    var_name='channel',  # Name the new column containing the former column labels.
    value_name='spend',  # Name the new column containing the spend values.
)

marketing_long.head(9)


,month,channel,spend
0,2025-01,digital,1200
1,2025-02,digital,1400
2,2025-03,digital,1500
3,2025-04,digital,1650
4,2025-05,digital,1725
5,2025-06,digital,1800
6,2025-01,catalogue,800
7,2025-02,catalogue,780
8,2025-03,catalogue,760


Once the data is long, the same measurement column can be grouped by channel.


In [ ]:
# Summarise spend after reshaping to long form.
channel_summary = (
    marketing_long.groupby('channel', as_index=False)  # Return channel as an ordinary column.
    .agg(
        total_spend=('spend', 'sum'),             # Add spend across all months.
        average_monthly_spend=('spend', 'mean'),  # Calculate mean monthly spend.
    )
    .sort_values('total_spend', ascending=False)
)

channel_summary


,channel,total_spend,average_monthly_spend
1,digital,9275,1545.833333
0,catalogue,4470,745.000000
2,events,4060,676.666667


### 3.2 Long to wide with `pivot()`

Each month–channel pair occurs once, so `pivot()` can reconstruct the original layout without aggregation.


In [ ]:
# Reconstruct one column per marketing channel.
marketing_pivot = (
    marketing_long.pivot(
        index='month',      # Use unique months as row labels.
        columns='channel',  # Create one column for each channel.
        values='spend',     # Fill the reshaped table with spend values.
    )
    .reset_index()  # Return month to an ordinary column.
)
marketing_pivot.columns.name = None

marketing_pivot


,month,catalogue,digital,events
0,2025-01,800,1200,500
1,2025-02,780,1400,620
2,2025-03,760,1500,650
3,2025-04,730,1650,720
4,2025-05,710,1725,760
5,2025-06,690,1800,810


### 3.3 Aggregate and reshape with `pivot_table()`

Create a quarter label so several months belong to each quarter. `pivot_table()` then adds the monthly rows while it reshapes them.


In [ ]:
# Add a calendar-quarter label from the monthly text values.
marketing_long['quarter'] = (
    pd.to_datetime(marketing_long['month'])  # Parse the month text as datetime values.
    .dt.to_period('Q')                       # Convert each month to its calendar quarter.
    .astype(str)                             # Store quarter labels such as 2025Q1 as text.
)

# Aggregate monthly values into a quarter-by-channel matrix.
quarterly_marketing = marketing_long.pivot_table(
    index='quarter',    # Use quarters as row labels.
    columns='channel',  # Create one column for each channel.
    values='spend',     # Summarise the spend column.
    aggfunc='sum',      # Add monthly values within each quarter-channel group.
    fill_value=0,       # Replace absent combinations with zero when appropriate.
)

quarterly_marketing


channel,catalogue,digital,events
quarter,,,
2025Q1,2340,4100,1770
2025Q2,2130,5175,2290


The same quarterly marketing matrix can be produced by grouping and unstacking. Comparing the results is a useful validation check when learning equivalent pandas workflows.


In [ ]:
# Aggregate first, then move channel labels into columns.
quarterly_via_groupby = (
    marketing_long.groupby(['quarter', 'channel'])['spend']  # Produce a two-level grouped Series.
    .sum()                             # Aggregate each quarter-channel combination.
    .unstack('channel', fill_value=0)  # Move channel labels from the index into columns.
)

print('groupby() + unstack() matches pivot_table():', quarterly_via_groupby.equals(quarterly_marketing))
quarterly_via_groupby


groupby() + unstack() matches pivot_table(): True


channel,catalogue,digital,events
quarter,,,
2025Q1,2340,4100,1770
2025Q2,2130,5175,2290


## 4. Reshaping with `stack()` and `unstack()`

`stack()` moves column labels into the row index. `unstack()` moves an index level back into columns. These methods are especially useful with a `MultiIndex`.

Unlike `melt()`, which returns named columns for the variable and value, `stack()` usually returns a `Series` with hierarchical index labels.


In [16]:
# Place month in the index, then move channel columns into a second index level.
marketing_indexed = marketing_wide.set_index('month')  # Keep month as the first index level.
stacked_spend = marketing_indexed.stack().rename('spend')  # Move channel columns into a second index level.
stacked_spend.index.names = ['month', 'channel']

print('Stacked Series with a MultiIndex:')
display(stacked_spend.head(9).to_frame())
print('Index level names:', stacked_spend.index.names)


Stacked Series with a MultiIndex:


spend
month   channel         
2025-01 digital     1200
        catalogue    800
        events       500
2025-02 digital     1400
        catalogue    780
        events       620
2025-03 digital     1500
        catalogue    760
        events       650

Index level names: ['month', 'channel']


Unstack the `channel` index level to recover a wide table. The result keeps `month` as its row index.


In [17]:
# Move the channel index level back into columns.
unstacked_spend = stacked_spend.unstack('channel', fill_value=0)  # Move channel labels back into columns.
unstacked_spend


channel,digital,catalogue,events
month,,,
2025-01,1200,800,500
2025-02,1400,780,620
2025-03,1500,760,650
2025-04,1650,730,720
2025-05,1725,710,760
2025-06,1800,690,810


## 5. Exercises

### 5.1 Exercise 1: Calculate each quarter's share of its state total

Use `groupby()` with `transform('sum')` to add the state total beside every original row. Then calculate `target_share` as the row target divided by the state total.

Try the task before checking the worked solution below.


In [18]:
# Worked solution: transform returns one result for every original row.
target_shares = targets.copy()
target_shares['state_total'] = (
    target_shares.groupby('state')['sales_target'].transform('sum')  # Repeat each state total beside its original rows.
)
target_shares['target_share'] = (
    target_shares['sales_target'] / target_shares['state_total']
)

target_shares.sort_values(['state', 'quarter'])


,state,quarter,sales_target,state_total,target_share
2,NSW,2025Q1,5050,10600,0.476415
3,NSW,2025Q2,5550,10600,0.523585
6,QLD,2025Q1,4800,10100,0.475248
7,QLD,2025Q2,5300,10100,0.524752
4,VIC,2025Q1,4950,10400,0.475962
5,VIC,2025Q2,5450,10400,0.524038
0,WA,2025Q1,4800,10100,0.475248
1,WA,2025Q2,5300,10100,0.524752


### 5.2 Exercise 2: Check a round-trip reshape

Melt the marketing table, pivot it back to wide form, and check that the reconstructed table equals the original after the columns are placed in the same order.


In [19]:
# Worked solution: wide -> long -> wide.
exercise_long = marketing_wide.melt(
    id_vars='month',  # Keep month fixed during the wide-to-long reshape.
    var_name='channel',  # Store the former column labels in channel.
    value_name='spend',  # Store the former cell values in spend.
)
exercise_wide = (
    exercise_long.pivot(index='month', columns='channel', values='spend')  # Rebuild one column per channel.
    .reset_index()[marketing_wide.columns]  # Restore month and the original column order.
)
exercise_wide.columns.name = None

print('Round-trip reconstruction matches the original:', exercise_wide.equals(marketing_wide))
exercise_wide


Round-trip reconstruction matches the original: True


,month,digital,catalogue,events
0,2025-01,1200,800,500
1,2025-02,1400,780,620
2,2025-03,1500,760,650
3,2025-04,1650,730,720
4,2025-05,1725,710,760
5,2025-06,1800,690,810


## 6. Summary

Method selection:

| Goal | Recommended method |
| --- | --- |
| match related rows using keys | `merge()` |
| append compatible extracts | `pd.concat()` |
| calculate summaries for categories | `groupby().agg()` |
| select and reshape grouped hierarchical labels | `xs()`, `.loc`, and `unstack()` |
| convert named measure columns to tidy rows | `melt()` |
| reshape unique long-form observations to columns | `pivot()` |
| aggregate and reshape in one operation | `pivot_table()` |
| move labels between columns and index levels | `stack()` / `unstack()` |

